# RTFM + FFTFormer Ablation Experiments
Runs 5 configurations on UCF-Crime: baseline + mods 1, 1-2, 1-3, 1-2-3-4

## Step 1 — Clone repository

In [ ]:
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/rtfm-ucf-replication'  # <-- update this
REPO_NAME       = 'rtfm-ucf-replication'

import os
if not os.path.isdir(REPO_NAME):
    !git clone {GITHUB_REPO_URL}
else:
    print(f'{REPO_NAME} already cloned — pulling latest')
    !git -C {REPO_NAME} pull

%cd {REPO_NAME}

## Step 2 — Install dependencies

In [ ]:
!pip install -q scikit-learn matplotlib tqdm scipy visdom

## Step 3 — Detect environment

In [ ]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB  = 'google.colab' in sys.modules

if IS_KAGGLE:
    SEARCH_ROOT = '/kaggle/input'
elif IS_COLAB:
    SEARCH_ROOT = '/content/drive/MyDrive'
else:
    SEARCH_ROOT = os.path.expanduser('~')

print(f'Kaggle: {IS_KAGGLE} | Colab: {IS_COLAB}')
print(f'Search root: {SEARCH_ROOT}')

## Step 4 — Mount Google Drive (Colab only)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not Colab — skipping Drive mount')

## Step 5 — Discover feature file paths

In [ ]:
import glob

def locate(sentinel, search_root=SEARCH_ROOT):
    """Return the directory containing `sentinel`, searching recursively."""
    hits = glob.glob(os.path.join(search_root, '**', sentinel), recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    return None

UCF_TEST_FEAT_DIR  = locate('Abuse028_x264_i3d.npy')
UCF_TRAIN_FEAT_DIR = locate('Abuse001_x264_i3d.npy')

# ---------- Manual override (uncomment if auto-detect fails) ----------
# UCF_TEST_FEAT_DIR  = '/kaggle/input/YOUR_DATASET/UCF_Test_ten_crop_i3d'
# UCF_TRAIN_FEAT_DIR = '/kaggle/input/YOUR_DATASET/UCF_Train_ten_crop_i3d'
# ----------------------------------------------------------------------

print(f'Test  features : {UCF_TEST_FEAT_DIR}')
print(f'Train features : {UCF_TRAIN_FEAT_DIR}')

if UCF_TEST_FEAT_DIR is None or UCF_TRAIN_FEAT_DIR is None:
    print('\n[WARNING] One or both feature directories not found.')
    print('Directory tree under SEARCH_ROOT:')
    !find {SEARCH_ROOT} -maxdepth 5 -type d 2>/dev/null | head -60
else:
    print('\nContents of test dir:')
    !ls {UCF_TEST_FEAT_DIR} | head -10
    print('\nContents of train dir:')
    !ls {UCF_TRAIN_FEAT_DIR} | head -10

## Step 6 — Rewrite .list files with correct paths

In [ ]:
assert UCF_TEST_FEAT_DIR  is not None, 'Set UCF_TEST_FEAT_DIR manually in Step 5'
assert UCF_TRAIN_FEAT_DIR is not None, 'Set UCF_TRAIN_FEAT_DIR manually in Step 5'

def rewrite_list(list_path, new_feat_dir):
    with open(list_path) as f:
        lines = f.readlines()
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            new_lines.append(line)
            continue
        basename = os.path.basename(parts[0])
        new_path = os.path.join(new_feat_dir, basename)
        rest = ' '.join(parts[1:]) if len(parts) > 1 else ''
        new_lines.append((new_path + ' ' + rest).strip() + '\n')
    with open(list_path, 'w') as f:
        f.writelines(new_lines)
    print(f'Rewrote {list_path}  ({len(new_lines)} entries)')

rewrite_list('list/ucf-i3d-test.list', UCF_TEST_FEAT_DIR)
rewrite_list('list/ucf-i3d.list',      UCF_TRAIN_FEAT_DIR)

with open('list/ucf-i3d-test.list') as f:
    first_test = f.readline().strip().split()[0]
with open('list/ucf-i3d.list') as f:
    first_train = f.readline().strip().split()[0]

!ls -lh "{first_test}"
!ls -lh "{first_train}"

## Step 7 — Generate ground-truth labels

In [ ]:
GT_PATH = 'list/gt-ucf-local.npy'
if os.path.exists(GT_PATH):
    print(f'{GT_PATH} already exists — skipping generation')
else:
    from make_gt_ucf_local import main as _generate_gt
    _generate_gt()
    print('GT generated')

import numpy as np
gt = np.load(GT_PATH)
print(f'GT shape: {gt.shape}  |  positives: {gt.sum()}')

## Step 8 — Imports

In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm

from model       import Model
from dataset     import Dataset
from train       import train
from test_10crop import test
from utils       import Visualizer, save_best_record
from config      import Config
import option

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Step 9 — Experiment configuration

In [ ]:
MAX_EPOCH   = 100
BATCH_SIZE  = 32

# Each entry: (label, mod_string)
EXPERIMENTS = [
    ('Baseline',     ''),
    ('Mod 1',        '1'),
    ('Mod 1+2',      '1,2'),
    ('Mod 1+2+3',    '1,2,3'),
    ('Mod 1+2+3+4',  '1,2,3,4'),
]

## Step 10 — Helper functions

In [ ]:
def make_args(mod_str='', max_epoch=MAX_EPOCH, batch_size=BATCH_SIZE):
    argv = ['--max-epoch', str(max_epoch), '--batch-size', str(batch_size)]
    if mod_str:
        argv += ['--mod', mod_str]
    return option.parser.parse_args(argv)


def mod_tag(mod_str):
    return 'mods_' + mod_str.replace(',', '') if mod_str else 'baseline'


def run_experiment(label, mod_str):
    print(f'\n{"="*60}')
    print(f'  Experiment: {label}  (--mod "{mod_str}")')
    print(f'{"="*60}')

    args   = make_args(mod_str)
    config = Config(args)

    active_mods = set()
    if mod_str:
        for m in mod_str.split(','):
            m = m.strip()
            if m.isdigit():
                active_mods.add(int(m))

    tag = mod_tag(mod_str)
    args.model_name = f'rtfm_ucf_{tag}_'

    !mkdir -p ckpt

    train_nloader = DataLoader(
        Dataset(args, test_mode=False, is_normal=True),
        batch_size=args.batch_size, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True)
    train_aloader = DataLoader(
        Dataset(args, test_mode=False, is_normal=False),
        batch_size=args.batch_size, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True)
    test_loader   = DataLoader(
        Dataset(args, test_mode=True),
        batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

    model = Model(args.feature_size, args.batch_size, active_mods=active_mods).to(DEVICE)

    w_quant_params = [p for n, p in model.named_parameters() if 'W_quant' in n]
    other_params   = [p for n, p in model.named_parameters() if 'W_quant' not in n]
    if w_quant_params:
        optimizer = optim.Adam([
            {'params': other_params,   'lr': config.lr[0], 'weight_decay': 0.005},
            {'params': w_quant_params, 'lr': config.lr[0] * 2, 'weight_decay': 0.0},
        ])
    else:
        optimizer = optim.Adam(model.parameters(), lr=config.lr[0], weight_decay=0.005)

    viz = Visualizer(env=f'rtfm_{tag}', use_incoming_socket=False)

    auc_history    = []   # ROC AUC every epoch
    pr_auc_history = []   # PR  AUC every epoch
    best_AUC       = -1

    for step in tqdm(range(1, args.max_epoch + 1), desc=label, dynamic_ncols=True):
        if step > 1 and config.lr[step - 1] != config.lr[step - 2]:
            for pg in optimizer.param_groups:
                pg['lr'] = config.lr[step - 1]

        if (step - 1) % len(train_nloader) == 0:
            loadern_iter = iter(train_nloader)
        if (step - 1) % len(train_aloader) == 0:
            loadera_iter = iter(train_aloader)

        train(loadern_iter, loadera_iter, model, args.batch_size, optimizer, viz, DEVICE)

        # test() now returns 6 values: auc, pr_auc, fpr, tpr, precision, recall
        auc, pr_auc, fpr, tpr, precision, recall = test(test_loader, model, args, viz, DEVICE)
        auc_history.append(auc)
        pr_auc_history.append(pr_auc)

        if auc > best_AUC:
            best_AUC = auc
            torch.save(model.state_dict(), f'ckpt/{args.model_name}best-i3d.pkl')
            save_best_record(
                {'epoch': list(range(1, step + 1)), 'test_AUC': auc_history},
                f'{args.model_name}best-AUC.txt')
            np.save(f'{args.model_name}fpr.npy',       fpr)
            np.save(f'{args.model_name}tpr.npy',       tpr)
            np.save(f'{args.model_name}precision.npy', precision)
            np.save(f'{args.model_name}recall.npy',    recall)
            tqdm.write(f'  *** New best AUC: {best_AUC:.4f}  (step {step}) ***')

    torch.save(model.state_dict(), f'ckpt/{args.model_name}final.pkl')

    from sklearn.metrics import auc as sk_auc
    best_PR_AUC = sk_auc(
        np.load(f'{args.model_name}recall.npy',    allow_pickle=True),
        np.load(f'{args.model_name}precision.npy', allow_pickle=True))

    return {
        'label':          label,
        'tag':            tag,
        'auc_history':    auc_history,
        'pr_auc_history': pr_auc_history,
        'best_AUC':       best_AUC,
        'best_PR_AUC':    best_PR_AUC,
        'fpr':            np.load(f'{args.model_name}fpr.npy',       allow_pickle=True),
        'tpr':            np.load(f'{args.model_name}tpr.npy',       allow_pickle=True),
        'precision':      np.load(f'{args.model_name}precision.npy', allow_pickle=True),
        'recall':         np.load(f'{args.model_name}recall.npy',    allow_pickle=True),
    }

## Step 11 — Run all experiments

In [ ]:
results = []
for label, mod_str in EXPERIMENTS:
    r = run_experiment(label, mod_str)
    results.append(r)
    print(f'{label}  best ROC-AUC={r["best_AUC"]:.4f}  best PR-AUC={r["best_PR_AUC"]:.4f}')

## Plots — ROC AUC and PR AUC per epoch

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for r in results:
    lbl = r['label']
    ax1.plot(r['auc_history'],    label=f"{lbl} (best={r['best_AUC']:.4f})")
    ax2.plot(r['pr_auc_history'], label=f"{lbl} (best={r['best_PR_AUC']:.4f})")

ax1.set_xlabel('Epoch'); ax1.set_ylabel('ROC AUC')
ax1.set_title('ROC AUC per Epoch'); ax1.legend(); ax1.grid(True)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('PR AUC')
ax2.set_title('PR AUC per Epoch');  ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('auc_curves.png', dpi=150)
plt.show()
print('Saved auc_curves.png')

## Plots — ROC curves (best checkpoint)

In [ ]:
from sklearn.metrics import auc as sk_auc

fig, ax = plt.subplots(figsize=(7, 7))
for r in results:
    roc_auc = sk_auc(r['fpr'], r['tpr'])
    ax.plot(r['fpr'], r['tpr'], label=f"{r['label']} AUC={roc_auc:.4f}")
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves (best epoch)')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()
print('Saved roc_curves.png')

## Plots — Precision-Recall curves (best checkpoint)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for r in results:
    pr_auc = sk_auc(r['recall'], r['precision'])
    ax.plot(r['recall'], r['precision'], label=f"{r['label']} AP={pr_auc:.4f}")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (best epoch)')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150)
plt.show()
print('Saved pr_curves.png')

## Plots — Best AUC bar chart

In [ ]:
labels      = [r['label']      for r in results]
best_roc    = [r['best_AUC']   for r in results]
best_pr     = [r['best_PR_AUC'] for r in results]

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], best_roc, width, label='ROC AUC', color='steelblue',  edgecolor='black')
bars2 = ax.bar([i + width/2 for i in x], best_pr,  width, label='PR AUC',  color='darkorange', edgecolor='black')

for bar, val in list(zip(bars1, best_roc)) + list(zip(bars2, best_pr)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('AUC'); ax.set_title('Best AUC by Configuration')
ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('best_auc_bar.png', dpi=150)
plt.show()
print('Saved best_auc_bar.png')

## (Optional) Re-load results without retraining
Run this cell to regenerate plots from saved `.npy` files.

In [ ]:
from sklearn.metrics import auc as sk_auc

def load_results(experiments):
    loaded = []
    for label, mod_str in experiments:
        tag  = mod_tag(mod_str)
        name = f'rtfm_ucf_{tag}_'
        try:
            fpr       = np.load(f'{name}fpr.npy',       allow_pickle=True)
            tpr       = np.load(f'{name}tpr.npy',       allow_pickle=True)
            precision = np.load(f'{name}precision.npy', allow_pickle=True)
            recall    = np.load(f'{name}recall.npy',    allow_pickle=True)
            best_AUC    = sk_auc(fpr, tpr)
            best_PR_AUC = sk_auc(recall, precision)
            loaded.append({
                'label': label, 'tag': tag,
                'auc_history': [], 'pr_auc_history': [],
                'best_AUC': best_AUC, 'best_PR_AUC': best_PR_AUC,
                'fpr': fpr, 'tpr': tpr,
                'precision': precision, 'recall': recall,
            })
            print(f'Loaded {label}  ROC-AUC={best_AUC:.4f}  PR-AUC={best_PR_AUC:.4f}')
        except FileNotFoundError:
            print(f'Missing files for {label} — skipping')
    return loaded

# Uncomment to use:
# results = load_results(EXPERIMENTS)